In [ ]:
# !pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import re
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection import cross_val_score

In [ ]:
# load data

df = pd.read_csv("startup_idea_validator_dataset.csv")
df.sample()

,id,sector,idea_title,idea_description,goals,objectives,employee_size_input,risk_level,already_in_market,competing_companies,...,funding_stage,target_market_size,revenue_model,time_to_market_months,estimated_initial_investment_usd,geographic_focus,b2b_or_b2c,tech_complexity,regulatory_risk,unique_value_proposition
6913,90967cbf-ac83-48b4-a691-177eff3272ef,LegalTech,LegalTech Idea #6914,Compliance automation tool for GDPR and data p...,Zero compliance violations,"1000 companies, continuous monitoring",1-10,High,No,None identified,...,Series B,$100B+,Enterprise License,8,<$50K,North America,B2B2C,High,Low,10x cheaper than incumbents


In [ ]:
# USE ONLY SAFE COLUMNS

df = df[['idea_description','sector','market_saturation','tech_complexity','funding_stage']]
df.dropna(inplace=True)

In [ ]:
# CREATE REALISTIC TARGET (ADD NOISE)

def create_risk(row):
  score = 0
  if row['market_saturation'] in ['High','Very High']:
    score += 2
  if row['tech_complexity'] in ['High','Very High']:
    score += 2
  if row['funding_stage'] in ['Bootstrapped','Pre-Seed']:
    score += 1

  # ADD RANDOMNESS
  score += np.random.choice([0,1], p=[0.7,0.3])

  if score <= 2:
    return "Low"
  elif score <= 4:
    return "Medium"
  else:
    return "High"
df['risk'] = df.apply(create_risk,axis=1)

In [ ]:
# CLEAN TEXT

def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]','',text)
  return text
df['text'] = (df['idea_description'] + " " + df['sector']).apply(clean_text)

In [ ]:
# TF-IDF

vectorizer = TfidfVectorizer(stop_words='english',max_features=3000,ngram_range=(1,2))
X_text = vectorizer.fit_transform(df['text'])

In [ ]:
# ENCODE FEATURES

le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

df['ms'] = le1.fit_transform(df['market_saturation'])
df['tc'] = le2.fit_transform(df['tech_complexity'])
df['fs'] = le3.fit_transform(df['funding_stage'])

In [ ]:
# COMBINE FEATURES

X = sp.hstack([X_text,df[['ms','tc','fs']].values])

In [ ]:
# TARGET

le = LabelEncoder()
y = le.fit_transform(df['risk'])

In [ ]:
# SPLIT

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
# MODEL

model = DecisionTreeClassifier(max_depth=6,min_samples_split=10,min_samples_leaf=5,random_state=42)
model.fit(X_train,y_train)

DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, min_samples_split=10,
                       random_state=42)

In [ ]:
# EVALUATION

y_pred = model.predict(X_test)
print("Accuracy : ",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

Accuracy :  0.8345
              precision    recall  f1-score   support

           0       0.94      0.64      0.76       314
           1       0.82      0.99      0.90       894
           2       0.83      0.73      0.78       792

    accuracy                           0.83      2000
   macro avg       0.86      0.79      0.81      2000
weighted avg       0.84      0.83      0.83      2000



In [ ]:
import joblib

In [ ]:
joblib.dump({
    "model":model,
    "vectorizer":vectorizer,
    "label_encoder":le,
    "ms_encoder":le1,
    "tc_encoder":le2,
    "fs_encoder":le3
},"best.pkl")

print("All saved in best.pkl")

All saved in best.pkl


In [ ]:
# ================================
# EXTRA FEATURES (UPGRADE)
# ================================

# Confidence score
probs = model.predict_proba(X_test)
confidence = np.max(probs, axis=1)

# Convert numeric prediction back to label
pred_labels = le.inverse_transform(y_pred)

# Risk message
def risk_message(risk):
    if risk == "Low":
        return "Startup idea has good potential ✅"
    elif risk == "Medium":
        return "Moderate risk, needs validation ⚠"
    else:
        return "High risk, rethink strategy ❌"

# Team suggestion
def suggest_team(tc_value):
    if tc_value == "Low":
        return "2-4 members"
    elif tc_value == "Medium":
        return "4-6 members"
    else:
        return "6-10 members"

# Show some sample outputs
for i in range(5):
    print("\n🔹 Startup Idea Result")
    print("Risk:", pred_labels[i])
    print("Confidence:", round(confidence[i],2))
    print("Message:", risk_message(pred_labels[i]))


🔹 Startup Idea Result
Risk: Low
Confidence: 0.74
Message: Startup idea has good potential ✅

🔹 Startup Idea Result
Risk: Low
Confidence: 1.0
Message: Startup idea has good potential ✅

🔹 Startup Idea Result
Risk: Medium
Confidence: 0.74
Message: Moderate risk, needs validation ⚠

🔹 Startup Idea Result
Risk: High
Confidence: 1.0
Message: High risk, rethink strategy ❌

🔹 Startup Idea Result
Risk: Low
Confidence: 0.7
Message: Startup idea has good potential ✅


In [ ]:
# CROSS VALIDATION

cv_scores = cross_val_score(model,X,y,cv=5)
print("CV Accuracy : ",np.mean(cv_scores))

CV Accuracy :  0.8335000000000001
